# RestroPulse restaurant market analysis

This notebook shares the cleaning, city analysis, category competition scoring, and model evaluation used by the Streamlit dashboard. Scores compare categories within the selected market. The opportunity index is only a screening proxy based on observed supply and reviews at existing businesses.

In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = next(parent for parent in (Path.cwd(), *Path.cwd().parents) if (parent / "src").is_dir())
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import plotly.express as px
from src.analysis import category_summary, city_summary, summary_metrics
from src.data_cleaning import load_csv
from src.ml_model import save_model_bundle, train_and_evaluate_models


In [2]:
restaurants, cleaning_report = load_csv(PROJECT_ROOT / "data" / "cleaned.csv")
print("Data quality:", cleaning_report)
print("Snapshot:", summary_metrics(restaurants))
restaurants.head()


Data quality: {'input_rows': 36889, 'retained_rows': 36889, 'missing_ratings_removed': 0, 'invalid_ratings_removed': 0, 'duplicate_ids_removed': 0, 'rows_removed': 0}
Snapshot: {'total_restaurants': 36889, 'average_rating': 3.473596465070888, 'total_reviews': 2245551, 'number_of_cities': 638}


,business_id,categories,city,name,review_count,stars,state,categories_clean,categories_display,category_count,location,log_review_count
0,LixFCMGKdptI8WRsjAl5cQ,"[u'Burgers', u'Fast Food', u'Restaurants']",Agincourt,McDonald's,6,2.5,ON,"[Burgers, Fast Food, Restaurants]","Burgers, Fast Food, Restaurants",3,"Agincourt, ON",1.945910
1,0Rni7ocMC_Lg2UH0lDeKMQ,"[u'Restaurants', u'Pizza', u'Italian', u'Sandw...",Ahwatukee,Barro's Pizza,62,3.5,AZ,"[Restaurants, Pizza, Italian, Sandwiches]","Restaurants, Pizza, Italian, Sandwiches",4,"Ahwatukee, AZ",4.143135
2,qJf61TR4Jq9Xph5RiXPS9A,"[u'Restaurants', u'Coffee & Tea', u'Creperies'...",Ahwatukee,Cupz N' Crepes,236,4.0,AZ,"[Restaurants, Coffee & Tea, Creperies, Food]","Restaurants, Coffee & Tea, Creperies, Food",4,"Ahwatukee, AZ",5.468060
3,eVv6cnwhabK8ig5Di2hXQQ,"[u'Italian', u'Restaurants', u'Pizza', u'Sandw...",Ahwatukee,Florencia Pizza Bistro,129,4.5,AZ,"[Italian, Restaurants, Pizza, Sandwiches]","Italian, Restaurants, Pizza, Sandwiches",4,"Ahwatukee, AZ",4.867534
4,-v8Z3mdbbPs1ljLziHr2DA,"[u'Gastropubs', u'Nightlife', u'Bars', u'Resta...",Ahwatukee,My Wine Cellar,158,4.0,AZ,"[Gastropubs, Nightlife, Bars, Restaurants, Win...","Gastropubs, Nightlife, Bars, Restaurants, Wine...",5,"Ahwatukee, AZ",5.068904


In [3]:
cities = city_summary(restaurants)
cities.head(15)


,location,city,state,restaurant_count,average_rating,total_reviews,median_reviews
0,"Toronto, ON",Toronto,ON,4613,3.452742,208310,21.0
1,"Las Vegas, NV",Las Vegas,NV,3852,3.438733,665573,52.0
2,"Phoenix, AZ",Phoenix,AZ,2427,3.435311,229919,35.0
3,"Montreal, QC",Montreal,QC,2381,3.692776,68188,12.0
4,"Charlotte, NC",Charlotte,NC,1701,3.390065,108965,28.0
5,"Pittsburgh, PA",Pittsburgh,PA,1501,3.520653,88866,27.0
6,"Edinburgh, EDH",Edinburgh,EDH,1010,3.832673,19295,11.0
7,"Cleveland, OH",Cleveland,OH,967,3.487590,47016,19.0
8,"Mississauga, ON",Mississauga,ON,960,3.393750,22801,12.5
9,"Stuttgart, BW",Stuttgart,BW,870,3.751149,15268,10.0


In [4]:
market = cities.iloc[0]["location"]
market_categories = category_summary(restaurants, location=market, minimum_restaurants=2)
print("Market selected:", market)
market_categories.head(15)


Market selected:

 Toronto, ON


,category,restaurant_count,restaurant_share_pct,total_reviews,review_share_pct,average_rating,review_weighted_rating,reviews_per_restaurant,competition_score,opportunity_index
0,Seafood,135,2.926512,9449,4.536028,3.611111,3.753731,69.992593,82.068345,42.823741
1,Tapas/Small Plates,49,1.062215,5726,2.748788,3.795918,3.783706,116.857143,81.978417,54.496403
2,Japanese,318,6.893562,23533,11.297105,3.548742,3.586156,74.003145,81.960432,40.629496
3,Tapas Bars,43,0.932148,4407,2.115597,3.720930,3.847402,102.488372,81.852518,54.478417
4,Canadian (New),397,8.606113,27401,13.153953,3.497481,3.550929,69.020151,81.294964,37.985612
5,Breakfast & Brunch,365,7.912421,22811,10.950506,3.516438,3.630091,62.495890,80.899281,36.474820
6,Noodles,32,0.693692,3585,1.720993,3.609375,3.787308,112.031250,80.296763,56.411871
7,Vegetarian,114,2.471277,7224,3.467908,3.714912,3.798173,63.368421,80.269784,41.780576
8,Bars,593,12.854975,37714,18.104748,3.461214,3.539296,63.598651,80.089928,35.809353
9,Vegan,73,1.582484,4458,2.140080,3.993151,3.959623,61.068493,80.035971,42.985612


In [5]:
competition_fig = px.bar(
    market_categories.nlargest(15, "competition_score").sort_values("competition_score"),
    x="competition_score", y="category", orientation="h",
    title=f"Relative competition in {market}",
    labels={"competition_score": "Competition score (0-100)", "category": "Category"},
)
opportunity_fig = px.bar(
    market_categories.nlargest(15, "opportunity_index").sort_values("opportunity_index"),
    x="opportunity_index", y="category", orientation="h",
    title=f"Opportunity proxy in {market}",
    labels={"opportunity_index": "Opportunity proxy (0-100)", "category": "Category"},
)
competition_fig.show()
opportunity_fig.show()


In [6]:
# Ratings are evaluated on a held-out test set before a final model is fit on all rows.
model_result = train_and_evaluate_models(restaurants)
pd.Series(model_result["metrics"], name="value")


train_rows                         29511.000000
test_rows                           7378.000000
mae                                    0.598705
rmse                                   0.757472
r2                                     0.090169
baseline_mae                           0.611819
mae_improvement_vs_baseline_pct        2.143480
silhouette_score                       0.301621
cluster_count                          4.000000
Name: value, dtype: float64

In [7]:
segments = model_result["segmented_restaurants"]
cluster_profiles = (
    segments.groupby("cluster")
    .agg(
        restaurants=("business_id", "nunique"),
        average_rating=("stars", "mean"),
        median_reviews=("review_count", "median"),
        average_categories=("category_count", "mean"),
    )
    .reset_index()
)
cluster_profiles


,cluster,restaurants,average_rating,median_reviews,average_categories
0,1,11230,3.999555,9.0,3.098664
1,2,9584,3.713272,89.5,3.334725
2,3,9815,2.501936,9.0,3.091594
3,4,6260,3.686581,35.0,6.984345


In [8]:
model_path = save_model_bundle(model_result, PROJECT_ROOT / "models")
print(f"Evaluated model bundle saved to {model_path}")


Evaluated model bundle saved to D:\Projects\RestroPulse – Restaurant Trends & Competition Analysis\missinglink-master\models\model.pkl


## Interpretation notes

- City identity is the observed city plus its state or province. A state is never substituted for a city.
- Restaurant category tags overlap, so category counts and shares are not mutually exclusive.
- Competition scores rank supply, review activity per restaurant, and ratings within the selected market.
- Review counts are a proxy for activity among existing Yelp listings. They do not measure restaurants that are missing from the dataset or predict business success.
- The linear regression holdout scores describe this random split. No opening date or time-series data is available to validate future rating forecasts.